In [3]:
# Ensure your VertexAI credentials are configured
import os
from google.cloud import aiplatform

KEYFILE_PATH = '/home/mc76728/repo/Coargus/vrag/cs391-project-11f0f788cfea.json'
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEYFILE_PATH
aiplatform.init(project='cs391-project')
import vertexai
vertexai.init(project='cs391-project')

In [4]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.0-flash-001", model_provider="google_vertexai")

In [5]:
from langchain_google_vertexai import VertexAIEmbeddings
from google.oauth2 import service_account

# Create credentials object
credentials = service_account.Credentials.from_service_account_file(KEYFILE_PATH)

# Pass credentials to the embeddings model
embeddings = VertexAIEmbeddings(
    model="text-embedding-004",
    credentials=credentials
)

In [7]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="/home/mc76728/repo/Coargus/vrag/artifacts/chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [8]:
import bs4
from langchain import hub
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/home/mc76728/repo/Coargus/vrag/merged_transcript.txt")

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

# Index chunks
_ = vector_store.add_documents(documents=all_splits)

# Define prompt for question-answering
prompt = hub.pull("rlm/rag-prompt")


# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


# Define application steps
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

# Compile application and test
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

/home/mc76728/repo/Coargus/vrag/.venv/lib/python3.10/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [9]:
response = graph.invoke({"question": "What is Task Decomposition?"})
print(response["answer"])

I'm sorry, but the provided context does not contain a definition of Task Decomposition. The context discusses pulesky decomposition, QR decomposition, and cholesky decomposition. It also mentions collaborative filtering and regression problems.

